# Transformer from Scratch
Seq2Seq Transformer with greedy decoding — scratch implementation in PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import pandas as pd
from torch.utils.data import DataLoader, Dataset

## Config

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d_model = 512
n_heads = 8
num_layers = 6
d_ff = 2048
max_len = 100
batch_size = 32
lr = 1e-4
epochs = 10
drop_prob = 0.1

## Load Data

In [ ]:

df = pd.read_csv("data.csv")  
src_texts = df["src"].tolist()
tgt_texts = df["tgt"].tolist()

## Tokenization

In [ ]:

def build_vocab(texts):
    vocab = {"<pad>":0, "<sos>":1, "<eos>":2}
    idx = 3
    for sent in texts:
        for word in sent.split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

def encode(text, vocab):
    return [vocab["<sos>"]] + [vocab[w] for w in text.split()] + [vocab["<eos>"]]

src_vocab = build_vocab(src_texts)
tgt_vocab = build_vocab(tgt_texts)

src_encoded = [encode(s, src_vocab) for s in src_texts]
tgt_encoded = [encode(t, tgt_vocab) for t in tgt_texts]

## Dataset & DataLoader

In [ ]:

class TranslationDataset(Dataset):
    def __init__(self, src, tgt):
        self.src = src
        self.tgt = tgt

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])

def pad_batch(batch):
    src, tgt = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src, padding_value=0)  # (T,B)
    tgt = nn.utils.rnn.pad_sequence(tgt, padding_value=0)
    return src, tgt

dataset = TranslationDataset(src_encoded, tgt_encoded)
loader = DataLoader(dataset, batch_size=batch_size, collate_fn=pad_batch)

## Positional Encoding

In [ ]:

def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            pe[pos, i] = math.sin(pos / (10000 ** (i/d_model)))
            pe[pos, i+1] = math.cos(pos / (10000 ** ((i+1)/d_model)))
    return pe.unsqueeze(0)  # (1,T,D)

PE = positional_encoding(max_len, d_model).to(device)

## Masks

In [ ]:

def create_padding_mask(seq):
    # seq: (T,B)
    mask = (seq != 0).transpose(0,1)  # (B,T)
    return mask.unsqueeze(1).unsqueeze(2)  # (B,1,1,T)

def create_tgt_mask(tgt):
    B = tgt.shape[1]
    T = tgt.shape[0]

    pad_mask = create_padding_mask(tgt)  # (B,1,1,T)

    causal = torch.tril(torch.ones(T, T)).bool().to(device)
    causal = causal.unsqueeze(0).unsqueeze(1)  # (1,1,T,T)

    return pad_mask & causal  # (B,1,T,T)

## Attention

In [ ]:

dropout = nn.Dropout(drop_prob)

def scaled_dot_product(q, k, v, mask=None):
    dk = q.size(-1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(dk)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attn = torch.softmax(scores, dim=-1)
    attn = dropout(attn)

    return torch.matmul(attn, v)

def multi_head_attention(x_q, x_k, x_v, Wq, Wk, Wv, Wo, mask=None):
    B, T, D = x_q.shape

    q = Wq(x_q)
    k = Wk(x_k)
    v = Wv(x_v)

    q = q.view(B, T, n_heads, D//n_heads).transpose(1,2)
    k = k.view(B, T, n_heads, D//n_heads).transpose(1,2)
    v = v.view(B, T, n_heads, D//n_heads).transpose(1,2)

    if mask is not None:
        mask = mask.unsqueeze(1)  

    out = scaled_dot_product(q, k, v, mask)

    out = out.transpose(1,2).contiguous().view(B, T, D)
    return Wo(out)

## Feed Forward

In [ ]:

def feed_forward(x, W1, W2):
    return W2(F.relu(W1(x)))

## Encoder & Decoder Layers

In [ ]:

def encoder_layer(x, params, src_mask):
    attn = multi_head_attention(x, x, x, *params["attn"], src_mask)
    x = params["ln1"](x + dropout(attn))

    ff = feed_forward(x, *params["ff"])
    x = params["ln2"](x + dropout(ff))
    return x

def decoder_layer(x, enc_out, params, tgt_mask, src_mask):
    attn1 = multi_head_attention(x, x, x, *params["self_attn"], tgt_mask)
    x = params["ln1"](x + dropout(attn1))

    # CROSS ATTENTION (decoder attends encoder)
    attn2 = multi_head_attention(x, enc_out, enc_out, *params["cross_attn"], src_mask)
    x = params["ln2"](x + dropout(attn2))

    ff = feed_forward(x, *params["ff"])
    x = params["ln3"](x + dropout(ff))
    return x

## Build Parameters

In [ ]:

def build_encoder_layer():
    return {
        "attn": [nn.Linear(d_model,d_model).to(device) for _ in range(4)],
        "ff": [nn.Linear(d_model,d_ff).to(device), nn.Linear(d_ff,d_model).to(device)],
        "ln1": nn.LayerNorm(d_model).to(device),
        "ln2": nn.LayerNorm(d_model).to(device),
    }

def build_decoder_layer():
    return {
        "self_attn": [nn.Linear(d_model,d_model).to(device) for _ in range(4)],
        "cross_attn": [nn.Linear(d_model,d_model).to(device) for _ in range(4)],
        "ff": [nn.Linear(d_model,d_ff).to(device), nn.Linear(d_ff,d_model).to(device)],
        "ln1": nn.LayerNorm(d_model).to(device),
        "ln2": nn.LayerNorm(d_model).to(device),
        "ln3": nn.LayerNorm(d_model).to(device),
    }

encoder_layers = [build_encoder_layer() for _ in range(num_layers)]
decoder_layers = [build_decoder_layer() for _ in range(num_layers)]

src_embed = nn.Embedding(len(src_vocab), d_model).to(device)
tgt_embed = nn.Embedding(len(tgt_vocab), d_model).to(device)
fc_out = nn.Linear(d_model, len(tgt_vocab)).to(device)

## Weight Initialisation

In [ ]:

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)

src_embed.apply(init_weights)
tgt_embed.apply(init_weights)
fc_out.apply(init_weights)

## Forward Pass

In [ ]:

def forward(src, tgt):
    src = src.to(device)
    tgt = tgt.to(device)

    src_mask = create_padding_mask(src)
    tgt_mask = create_tgt_mask(tgt)

    # embeddings + scaling
    src = src_embed(src) * math.sqrt(d_model)
    tgt = tgt_embed(tgt) * math.sqrt(d_model)

    # convert to (B,T,D)
    src = src.transpose(0,1)
    tgt = tgt.transpose(0,1)

    # add positional encoding
    src = dropout(src + PE[:, :src.size(1), :])
    tgt = dropout(tgt + PE[:, :tgt.size(1), :])

    # encoder
    x = src
    for layer in encoder_layers:
        x = encoder_layer(x, layer, src_mask)
    enc_out = x

    # decoder
    y = tgt
    for layer in decoder_layers:
        y = decoder_layer(y, enc_out, layer, tgt_mask, src_mask)

    return fc_out(y)

## Optimizer & Loss

In [ ]:

params = list(src_embed.parameters()) + list(tgt_embed.parameters()) + list(fc_out.parameters())

for layer in encoder_layers:
    for v in layer.values():
        if isinstance(v, list):
            for m in v: params += list(m.parameters())
        else:
            params += list(v.parameters())

for layer in decoder_layers:
    for v in layer.values():
        if isinstance(v, list):
            for m in v: params += list(m.parameters())
        else:
            params += list(v.parameters())

optimizer = torch.optim.Adam(params, lr=lr)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

## Training Loop

In [ ]:

for epoch in range(epochs):
    total_loss = 0

    for src, tgt in loader:
        tgt_input = tgt[:-1]
        tgt_output = tgt[1:]

        logits = forward(src, tgt_input)

        loss = loss_fn(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1).to(device)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss}")

## Inference

In [ ]:

rev_tgt_vocab = {v:k for k,v in tgt_vocab.items()}


def encode_input(sentence, vocab):
    tokens = [vocab["<sos>"]]
    for w in sentence.split():
        tokens.append(vocab.get(w, 0)) 
    tokens.append(vocab["<eos>"])
    return torch.tensor(tokens).unsqueeze(1)  


def translate(sentence, max_len=50):
    src = encode_input(sentence, src_vocab).to(device)

    # encoder pass
    src_mask = create_padding_mask(src)

    src_emb = src_embed(src) * math.sqrt(d_model)
    src_emb = src_emb.transpose(0,1)
    src_emb = dropout(src_emb + PE[:, :src_emb.size(1), :])

    enc_out = src_emb
    for layer in encoder_layers:
        enc_out = encoder_layer(enc_out, layer, src_mask)

    # decoder start
    tgt_tokens = [tgt_vocab["<sos>"]]

    for _ in range(max_len):
        tgt = torch.tensor(tgt_tokens).unsqueeze(1).to(device)  # (T,1)

        tgt_mask = create_tgt_mask(tgt)

        tgt_emb = tgt_embed(tgt) * math.sqrt(d_model)
        tgt_emb = tgt_emb.transpose(0,1)
        tgt_emb = dropout(tgt_emb + PE[:, :tgt_emb.size(1), :])

        dec_out = tgt_emb
        for layer in decoder_layers:
            dec_out = decoder_layer(dec_out, enc_out, layer, tgt_mask, src_mask)

        logits = fc_out(dec_out)  # (1,T,V)

        next_token = torch.argmax(logits[:, -1, :], dim=-1).item()
        tgt_tokens.append(next_token)

        if next_token == tgt_vocab["<eos>"]:
            break


    words = []
    for t in tgt_tokens[1:]:
        if t == tgt_vocab["<eos>"]:
            break
        words.append(rev_tgt_vocab.get(t, "<unk>"))

    return " ".join(words)

## Test

In [ ]:

print(translate("હા ભાઈ, આવી ગયા તમે પણ અહીં સુધી"))